# M04 — CTA Data Cleaning

M02 ki Bus aur Rail CSV files ko alag clean karte hain. Cells upar se neeche run karo.

## 1. Data load karo

In [1]:
import pandas as pd

bus = pd.read_csv(
    r"C:\UZZAM WORK\5th Semester\Artificial Intelligence & Expert Systems\Project\Transit_Ridership_M01_Setup\Transit_Ridership_M10_Reviewed\transit-ridership\backend\data\processed\bus_2022_2025.csv",
    dtype=str, keep_default_na=False
)
rail = pd.read_csv(
    r"C:\UZZAM WORK\5th Semester\Artificial Intelligence & Expert Systems\Project\Transit_Ridership_M01_Setup\Transit_Ridership_M10_Reviewed\transit-ridership\backend\data\processed\rail_2022_2025.csv",
    dtype=str, keep_default_na=False
)

datasets = {"Bus": bus, "Rail": rail}
ready_to_save = False

for name, df in datasets.items():
    print(f"{name}: {len(df):,} rows")

Bus: 167,648 rows
Rail: 209,472 rows


## 2. Values clean aur check karo

In [2]:
ready_to_save = False
problems_found = False

required = {
    "Bus": {"route", "date", "daytype", "rides"},
    "Rail": {"station_id", "stationname", "date", "daytype", "rides"}
}

for name, df in datasets.items():
    missing_columns = required[name] - set(df.columns)
    if missing_columns:
        raise ValueError(f"{name}: missing columns: {sorted(missing_columns)}")

    df = df.apply(lambda column: column.str.strip().replace("", pd.NA))
    df["date"] = pd.to_datetime(df["date"], format="mixed", errors="coerce")

    rides = df["rides"]
    has_commas = rides.str.fullmatch(r"\d{1,3}(?:,\d{3})+", na=False)
    rides = rides.where(~has_commas, rides.str.replace(",", "", regex=False))
    df["rides"] = pd.to_numeric(rides, errors="coerce")

    id_column = "route" if name == "Bus" else "station_id"
    invalid_rides = df["rides"].isna() | df["rides"].isin([float("inf"), float("-inf")])
    invalid_rides |= (df["rides"] < 0) | (df["rides"] % 1 != 0)

    checks = {
        "Missing route/station ID": df[id_column].isna(),
        "Missing or invalid date": df["date"].isna(),
        "Date outside 2022–2025": df["date"].notna() & ~df["date"].between("2022-01-01", "2025-12-31 23:59:59.999999999"),
        "Missing or unexpected day type": ~df["daytype"].isin(["W", "A", "U"]),
        "Missing or invalid rides": invalid_rides
    }

    print(f"\n{name} missing values:")
    print(df.isna().sum().to_string())
    print(f"\n{name} checks:")
    for label, mask in checks.items():
        count = int(mask.sum())
        print(f"  {label}: {count}")
        if count:
            problems_found = True

    datasets[name] = df

if problems_found:
    raise ValueError("Invalid data mila hai. Save karne se pehle investigate karo.")


Bus missing values:
route      0
date       0
daytype    0
rides      0

Bus checks:
  Missing route/station ID: 0
  Missing or invalid date: 0
  Date outside 2022–2025: 0
  Missing or unexpected day type: 0
  Missing or invalid rides: 0

Rail missing values:
station_id     0
stationname    0
date           0
daytype        0
rides          0

Rail checks:
  Missing route/station ID: 0
  Missing or invalid date: 0
  Date outside 2022–2025: 0
  Missing or unexpected day type: 0
  Missing or invalid rides: 0


## 3. Duplicate rows

In [ ]:
ready_to_save = False
conflicts_found = False

for name, df in datasets.items():
    df["date"] = df["date"].dt.normalize()
    df["rides"] = df["rides"].astype("int64")

    duplicates = int(df.duplicated().sum())
    df = df.drop_duplicates().copy()

    id_column = "route" if name == "Bus" else "station_id"
    conflicts = int(df.duplicated(subset=[id_column, "date"]).sum())
    print(f"{name}: {duplicates} exact duplicate rows removed; {len(df):,} rows remain")
    print(f"{name}: repeated ID + date rows: {conflicts}")

    if conflicts:
        conflicts_found = True
    datasets[name] = df

if conflicts_found:
    raise ValueError("Same ID aur date ki different rows mili hain. Pehle investigate karo.")

ready_to_save = True

Bus: 0 exact duplicate rows removed; 167,648 rows remain
Bus: repeated ID + date rows: 0
Rail: 0 exact duplicate rows removed; 209,472 rows remain
Rail: repeated ID + date rows: 0


## 4. Cleaned data preview

In [4]:
datasets["Bus"].head()

,route,date,daytype,rides
0,1,2022-01-03,W,385
1,1,2022-01-04,W,470
2,1,2022-01-05,W,289
3,1,2022-01-06,W,339
4,1,2022-01-07,W,233


In [5]:
datasets["Rail"].head()

,station_id,stationname,date,daytype,rides
0,40010,Austin-Forest Park,2022-01-01,A,231
1,40020,Harlem-Lake,2022-01-01,A,500
2,40030,Pulaski-Lake,2022-01-01,A,284
3,40040,Quincy/Wells,2022-01-01,A,493
4,40050,Davis,2022-01-01,A,502


## 5. Cleaned files save karo

Ye cell existing `bus_cleaned.csv` aur `rail_cleaned.csv` ko overwrite karegi. Agar unmein manual changes hain to pehle backup banao. Filtered input files change nahi hongi.

In [6]:
if not ready_to_save:
    raise ValueError("Save se pehle previous cells successfully run karo.")

datasets["Bus"].to_csv(
    r"C:\UZZAM WORK\5th Semester\Artificial Intelligence & Expert Systems\Project\Transit_Ridership_M01_Setup\Transit_Ridership_M10_Reviewed\transit-ridership\backend\data\processed\bus_cleaned.csv",
    index=False, date_format="%Y-%m-%d"
)
datasets["Rail"].to_csv(
    r"C:\UZZAM WORK\5th Semester\Artificial Intelligence & Expert Systems\Project\Transit_Ridership_M01_Setup\Transit_Ridership_M10_Reviewed\transit-ridership\backend\data\processed\rail_cleaned.csv",
    index=False, date_format="%Y-%m-%d"
)

print(f"Bus saved: {len(datasets['Bus']):,} rows")
print(f"Rail saved: {len(datasets['Rail']):,} rows")

Bus saved: 167,648 rows
Rail saved: 209,472 rows


Zero rides automatically error nahi hoti, isliye unhein retain kiya hai. Invalid essential values ya conflicting ID/date rows milne par notebook rukti hai.